In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
%run helperFunctions.ipynb


In [58]:
# 1. Define LSTM Autoencoder
def build_lstm_autoencoder(timesteps, features):
    # Encoder
    inputs = tf.keras.Input(shape=(timesteps, features), name='encoder_input')
    encoded = layers.LSTM(64, activation='relu', return_sequences=False)(inputs)
    encoded = layers.Dropout(0.1)(encoded)
    encoded = layers.Dense(32, activation='relu')(encoded)
    encoded = layers.Dropout(0.1)(encoded)
    # Latent Space
    latent = layers.Dense(16, activation='relu', name='latent')(encoded)
    
    # Decoder
    decoded = layers.Dense(32, activation='relu')(latent)
    decoded = layers.Dropout(0.1)(decoded)
    decoded = layers.RepeatVector(timesteps)(decoded)
    decoded = layers.Dropout(0.1)(decoded)
    decoded = layers.LSTM(64, activation='relu', return_sequences=True)(decoded)
    
    # Output
    outputs = layers.TimeDistributed(layers.Dense(features, activation='linear'))(decoded)
    
    # Build and Compile
    autoencoder = Model(inputs, outputs, name='LSTM_Autoencoder')
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder

In [59]:

# 2. Create Sliding Windows
def create_sliding_windows(data, window_size, step=1):
    num_samples, num_features = data.shape
    windows = []
    for start in range(0, num_samples - window_size + 1, step):
        end = start + window_size
        windows.append(data[start:end])
    return np.array(windows)


In [60]:
def train():
    """
    Trains the LSTM Autoencoder on the training dataset.
    """

    # Load training data
    df = getTrainingDataResistance()
    
    # Extract features
    X = df[["circ1.r_load.v", "circ1.r_load.i"]].astype(float).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    # Save the scaler for future use
    joblib.dump(scaler, './models/lstm_autoencoder.save')

    
    # Create sliding windows
    window_size = 3  # Adjust based on your data's temporal resolution
    step_size = 1
    X_windows = create_sliding_windows(X_normal_scaled, window_size, step=step_size)
    print(f"Training windows shape: {X_train_windows.shape}")
    
    # Define model parameters
    timesteps = window_size
    features = X_windows.shape[2]
    
    # Build the autoencoder
    autoencoder = build_lstm_autoencoder(timesteps, features)
    autoencoder.summary()
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

    
    # Train the autoencoder
    history = autoencoder.fit(
        X_windows, 
        X_windows, 
        epochs=20,
        batch_size=4,
        validation_split=0.1,
        #callbacks=[early_stop, reduce_lr],
        verbose=1
    )
  
    
    # Plot training and validation loss
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['loss'], label='Training Loss (MSE)')
    plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('LSTM Autencoder')
    plt.legend()
    plt.show()
    
    # Compute reconstruction errors on training data
    reconstructed_train = autoencoder.predict(X_train_windows)
    train_errors = np.mean(np.power(X_train_windows - reconstructed_train, 2), axis=(1,2))
    
    # Save training errors for threshold determination
   # joblib.dump(train_errors, 'train_errors.save')
   # print("Training errors saved to 'train_errors.save'.")
    
    # Save the trained model
    autoencoder.save('./models/lstm_autoencoder.h5')
    print("Autoencoder model saved to 'lstm_autoencoder.h5'.")
    return autoencoder


In [56]:
autoencoder = train()

read: ./data/training/Circuit_ScenarioDataGeneration_Restistance_res.csv


NameError: name 'joblib' is not defined

In [47]:
def test(autoencoder):
    """
    Tests the trained LSTM Autoencoder on the testing dataset and visualizes anomalies.
    """
    # File paths for testing data
    test_file_list = [
        #s"./data/Circuit_ScenarioDataGeneration_res.csv",
        "./data/Circuit_Scenario1_res.csv", 
        "./data/Circuit_Scenario2_res.csv", 
        "./data/Circuit_Scenario3_res.csv", 
        "./data/Circuit_Scenario4_res.csv",
        "./data/Circuit_Scenario5_res.csv",
        "./data/Circuit_Scenario6_res.csv",
        "./data/Circuit_Scenario7_res.csv",
        "./data/Circuit_Scenario8_res.csv",
        #"./data/Circuit_Scenario9_res.csv",
        #"./data/Circuit_Scenario10_res.csv",
        #"./data/Circuit_ScenarioVoltageDropsBat_res.csv",
        #"./data/Circuit_ScenarioVoltageSwitch_res.csv"
    ]
    
    # Load testing data
    df_test = readDatasInPanda(test_file_list)
    
    # Extract features
    X_test = df_test[["circ1.r_load.v", "circ1.r_load.i"]].astype(float).values
    
    # Load the saved scaler
   # scaler = joblib.load('scaler.save')
   # print("Scaler loaded from 'scaler.save'.")
    
    # Scale the testing data
  #  X_test_scaled = scaler.transform(X_test)
    
    # Create sliding windows
    window_size = 30  # Must match the training window size
    step_size = 1
    X_test_windows = create_sliding_windows(X_test, window_size, step=step_size)
    print(f"Testing windows shape: {X_test_windows.shape}")
    
    # Load the trained autoencoder
   
    #autoencoder = tf.keras.models.load_model('lstm_autoencoder.h5')
    print("Autoencoder model loaded from 'lstm_autoencoder.h5'.")
    
    # Compute reconstruction errors
    reconstructed_test = autoencoder.predict(X_test_windows)
    test_errors = np.mean(np.power(X_test_windows - reconstructed_test, 2), axis=(1,2))
    
    # Determine the threshold
    # Load training errors or set based on test data
    # Here, assuming you have saved training errors or set a percentile
    # For demonstration, we'll set the threshold as the 99th percentile of training reconstruction errors
    # You should ideally load these from training or define them appropriately
    
    # For this example, let's assume you have stored the threshold after training
    # If not, you need to compute it based on training data
    # Here, we'll set it manually (replace with actual value)
    
    # Example threshold (replace with the actual threshold used during training)
    # threshold = 0.5  # Placeholder value
    # Alternatively, if you have the training errors saved:
    # train_errors = joblib.load('train_errors.save')
    # threshold = np.percentile(train_errors, 99)
    
    # For this example, let's load the training errors from 'train_errors.save'
    # Ensure you modify the `train()` function to save training errors
    #try:
    #    train_errors = joblib.load('train_errors.save')
    #    threshold = np.percentile(train_errors, 99)
    #    print(f"Reconstruction Error Threshold (99th percentile): {threshold:.4f}")
   # except FileNotFoundError:
   #     print("Training errors not found. Please modify the train() function to save 'train_errors.save'.")
   #     return
    threshold = 500

    # Detect anomalies
    anomaly_mask = test_errors > threshold
    print(f"Number of anomalies detected: {np.sum(anomaly_mask)} / {len(test_errors)}")
    
    # Retrieve 'Anomalous' Labels from DataFrame
    # Create corresponding window labels based on the original DataFrame
    df_d_test = df_test.iloc[:len(X_test_windows)*step_size + window_size - 1].reset_index(drop=True)
    anomolous = isAnomalous(df_d_test)
    anomolous = np.array(anomolous)
    
    # Assign each window a label: True if any point in the window is anomalous
    window_labels = []
    for i in range(len(X_test_windows)):
        window_start = i * step_size
        window_end = window_start + window_size
        if window_end > len(anomolous):
            window_end = len(anomolous)
        window_anomalies = anomolous[window_start:window_end]
        window_label = window_anomalies.any()  # True if any point in window is anomalous
        window_labels.append(window_label)
    
    window_labels = np.array(window_labels)
    
    # Evaluate Performance (Optional)
    # If you have true labels, compute ROC-AUC
    auc_score = roc_auc_score(window_labels, test_errors)
    print(f"ROC-AUC Score: {auc_score:.4f}")
    
    # Plotting Reconstruction Error Histogram with Anomalies Highlighted
    normal_errors = test_errors[~anomaly_mask]
    anomalous_errors = test_errors[anomaly_mask]
    
    plt.figure(figsize=(10, 6))
    plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal', color='blue', edgecolor='black')
    plt.hist(anomalous_errors, bins=50, alpha=0.7, label='Anomalous', color='red', edgecolor='black')
    plt.axvline(x=threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.2f}')
    plt.xlabel("Reconstruction Error (MSE)")
    plt.ylabel("Count")
    plt.title("Histogram of Reconstruction Errors with Anomalies Highlighted")
    plt.legend()
    plt.show()
    
    # Enhanced Visualization with Seaborn
    error_df = pd.DataFrame({
        'Reconstruction_Error': test_errors,
        'Anomaly': anomaly_mask
    })
    
    plt.figure(figsize=(10, 6))
    sns.histplot(
        data=error_df[~error_df['Anomaly']], 
        x='Reconstruction_Error', 
        bins=50, 
        color='blue', 
        label='Normal', 
        kde=True, 
        stat="density", 
        alpha=0.6
    )
    sns.histplot(
        data=error_df[error_df['Anomaly']], 
        x='Reconstruction_Error', 
        bins=50, 
        color='red', 
        label='Anomalous', 
        kde=True, 
        stat="density", 
        alpha=0.6
    )
    plt.axvline(x=threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.2f}')
    plt.xlabel("Reconstruction Error (MSE)")
    plt.ylabel("Density")
    plt.title("Distribution of Reconstruction Errors with Anomalies Highlighted")
    plt.legend()
    plt.show()
    
    # Optional: Plot Reconstruction Errors Over Time with Anomalies Highlighted
    plt.figure(figsize=(15, 6))
    plt.plot(test_errors, label='Reconstruction Error', color='blue')
    plt.axhline(y=threshold, color='green', linestyle='--', linewidth=2, label='Threshold')
    plt.scatter(
        np.where(anomaly_mask)[0], 
        test_errors[anomaly_mask], 
        color='red', 
        label='Anomalies'
    )
    plt.xlabel("Window Index")
    plt.ylabel("Reconstruction Error (MSE)")
    plt.title("Reconstruction Errors Over Time with Anomalies Highlighted")
    plt.legend()
    plt.show()


In [48]:
test(autoencoder)

read: ./data/Circuit_Scenario1_res.csv
read: ./data/Circuit_Scenario2_res.csv
read: ./data/Circuit_Scenario3_res.csv
read: ./data/Circuit_Scenario4_res.csv
read: ./data/Circuit_Scenario5_res.csv
read: ./data/Circuit_Scenario6_res.csv
read: ./data/Circuit_Scenario7_res.csv
read: ./data/Circuit_Scenario8_res.csv
Testing windows shape: (1995, 30, 2)
Autoencoder model loaded from 'lstm_autoencoder.h5'.
63/63 [==============================] - 0s 2ms/step


ValueError: operands could not be broadcast together with shapes (1995,30,2) (1995,3,2) 